### Topic: How to Create Custom Tools (The 3 Methods)

### Agenda:

- 0. Introduction of Custom Tools

- 1. Custom Tools with @tool decorator and Practical Examples 

- 2. Custom Tools with StructuredTool and Pydantic

- 3. Custom Tools with BaseTool Subclass and Practical Examples 

- 4. Complete Summary of Tools


### 0. Introduction of Custom Tools

- A custom tools in a tool that we define ourself.

- Use them when:
    - We want to call our own APIs.
    - We want to encapsulate business logic.
    - We want the LLM to interact with our database, product or app.


- all tools are runnable.
    

### 1. Custom Tools with @tool decorator and Practical Examples 

In [ ]:

"""         Key Components of a Tool
        ===============================
- Every Tool in LangChain consists of four fundamental pieces:

┌─────────────────────────────────────────────────────────────┐
│                    ANATOMY OF A TOOL                        │
│                                                             │
│  1. NAME          → Unique identifier (e.g., "get_weather") │
│  2. DESCRIPTION   → Clear prompt telling the LLM WHEN and   │
│                     WHY to use this tool                    │
│  3. ARGS_SCHEMA   → Pydantic model defining expected inputs │
│                     types, and constraints                  │
│  4. RUN FUNCTION  → The actual Python code (sync / async)   │
│                     that executes the operation             │
└─────────────────────────────────────────────────────────────┘


- Steps of Custom tools
        # Step1: Create a function
        # Step2: add type hints
        # Step3: add tools decorator

"""

In [4]:
# Example 1: Custom tools with @tools decorator
from langchain_core.tools import tool

@tool
def multiply(a:int, b:int)-> int:
    """ Multiply two numbers """
    return a * b



result = multiply.invoke({"a": 5, "b": 5})

print(f"Result: {result}")

Result: 25


#### See the Components of a Tool

In [8]:
#  1. NAME 
print(f"Name of Tool: {multiply.name}")

# 2. DESCRIPTION
print(f"Description of Tool: {multiply.description}")

# 3. ARGS_SCHEMA
print(f"Argument Schema of Tool: {multiply.args}")

Name of Tool: multiply
Description of Tool: Multiply two numbers
Argument Schema of Tool: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [10]:
# What LLM see when we pass tools
print(f"LLM input for tool call: \n{multiply.args_schema.model_json_schema()}")

LLM input for tool call: 
{'description': 'Multiply two numbers ', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


### 2. Custom Tools with StructuredTool class and Pydantic 

- Definition:
    - A Structured Tool in LangChain is a special type of tool where the input to the tool follows a structured schema typically defined using a Pydantic model.

    - Dynamic programmatic tool creation

In [ ]:
# Example 1: Custom Tool with StructuredTool and Pydantic

# Import StructuredTool from its current module
from langchain_core.tools import StructuredTool

# Import Pydantic classes
from pydantic import BaseModel, Field


# --------------------------------------------------
# Step 1: Create the custom Python function
# --------------------------------------------------

def multiply_func(a: int, b: int) -> int:
    """
    Multiply two numbers and return the result.
    """
    return a * b


# --------------------------------------------------
# Step 2: Define the input schema
# --------------------------------------------------

class MultiplyInput(BaseModel):
    a: int = Field(
        ..., # This field is required and has no default value.
        description="First number to multiply"
    )

    b: int = Field(
        ...,
        description="Second number to multiply"
    )


# --------------------------------------------------
# Step 3: Convert the function into a StructuredTool
# --------------------------------------------------

multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)


# --------------------------------------------------
# Step 4: Invoke the tool
# --------------------------------------------------

result = multiply_tool.invoke({
    "a": 5,
    "b": 20
})


# --------------------------------------------------
# Step 5: Display the result
# --------------------------------------------------

print(f"Result: {result}")
print(f"Tool name: {multiply_tool.name}")
print(f"Description: {multiply_tool.description}")

Result: 100
Tool name: multiply
Description: Multiply two numbers


### 3. Custom Tools with BaseTool Subclass and Practical Examples 
- Definition:
    - BaseTool is the abstract base class for all tools in LangChain.

    - It defines the core structure and interface that any tool must follow,
    - whether it's a simple one-liner or a fully customized function.

    - All other tool types like @tool, StructuredTool are built on top of BaseTool

In [ ]:
from langchain.tools import BaseTool
from typing import Type


# arg schema using pydantic
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")


class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b


multiply_tool = MultiplyTool()



result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


In [ ]:
# ============================================================
# Example 2: Creating a Custom Tool by Inheriting BaseTool
# ============================================================

# Import BaseTool.
# BaseTool is the base class provided by LangChain for creating
# custom tools by defining our own tool class.
from langchain_core.tools import BaseTool

# Import BaseModel and Field from Pydantic.
# BaseModel is used to define the input schema of our tool.
# Field is used to provide metadata such as descriptions.
from pydantic import BaseModel, Field

# Type is used to specify the type of args_schema.
from typing import Type


# ============================================================
# Step 1: Define the Tool Input Schema
# ============================================================

class MultiplyInput(BaseModel):
    """
    Defines the input structure required by the MultiplyTool.

    The tool expects two integer values:
        a -> first number
        b -> second number
    """

    # '...' means this field is required.
    # The description helps LangChain/LLM understand what
    # this argument represents.
    a: int = Field(
        ...,
        description="The first number to multiply"
    )

    # The second required input.
    b: int = Field(
        ...,
        description="The second number to multiply"
    )


# ============================================================
# Step 2: Create a Custom Tool Class
# ============================================================

class MultiplyTool(BaseTool):
    """
    Custom LangChain tool for multiplying two numbers.

    By inheriting from BaseTool, this class becomes a
    LangChain-compatible tool.
    """

    # --------------------------------------------------------
    # Tool name
    # --------------------------------------------------------
    # This is the name LangChain/LLM will use to identify
    # this tool.
    name: str = "multiply"

    # --------------------------------------------------------
    # Tool description
    # --------------------------------------------------------
    # This tells the LLM what the tool does.
    description: str = "Multiply two numbers"

    # --------------------------------------------------------
    # Tool input schema
    # --------------------------------------------------------
    # Connect the Pydantic schema to the tool.
    #
    # This tells LangChain:
    # "This tool expects 'a' and 'b' as inputs."
    args_schema: Type[BaseModel] = MultiplyInput

    # --------------------------------------------------------
    # Tool execution logic
    # --------------------------------------------------------
    # _run() contains the actual operation performed by
    # the tool.
    #
    # LangChain calls this method when the tool is invoked.
    def _run(self, a: int, b: int) -> int:

        # Multiply the two input numbers and return the result.
        return a * b


# ============================================================
# Step 3: Create an Instance of the Custom Tool
# ============================================================

# Create an object from our MultiplyTool class.
multiply_tool = MultiplyTool()


# ============================================================
# Step 4: Invoke the Tool
# ============================================================

# Pass the required arguments to the tool.
#
# 'a' = 3
# 'b' = 3
#
# LangChain validates these inputs using MultiplyInput
# and then calls the _run() method.
result = multiply_tool.invoke({
    "a": 3,
    "b": 3
})


# ============================================================
# Step 5: Display the Result
# ============================================================

# Display the multiplication result.
print(f"Result: {result}")

# Display the tool name.
print(f"Tool Name: {multiply_tool.name}")

# Display the tool description.
print(f"Description: {multiply_tool.description}")

# Display the tool's argument schema.
print(f"Arguments Schema: {multiply_tool.args}")

### 4. Complete Summary of Tools

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│                   TOOLS IN LANGCHAIN                             │
│                                                                  │
│  WHAT:  Functions or capabilities that an LLM can invoke to      │
│         extend beyond text generation (APIs, databases, code).   │
│                                                                  │
│  WHY:   LLMs alone cannot:                                       │
│         - Perform real-world actions                             │
│         - Call external APIs                                     │
│         - Query databases                                        │
│         - Do reliable math                                       │
│         - Take decisions from user requests                      │
│                                                                  │
│  HOW IT WORKS INTERNALLY:                                        │
│    1. LLM receives prompt with tool descriptions                 │
│    2. LLM decides: do I need a tool?                             │
│    3. LLM generates tool call (name + parameters)                │
│    4. Framework parses and executes the tool                     │
│    5. Result fed back to LLM                                     │
│    6. LLM either calls another tool or answers                   │
│                                                                  │
│  BASIC SYNTAX:                                                   │
│    @tool                                                         │
│    def my_tool(param: type) -> type:                             │
│        """Description of what the tool does."""                  │
│        return result                                             │
│                                                                  │
│  TOOL ATTRIBUTES:                                                │
│    name          → Name of the tool                              │
│    description   → What the tool does (extracted from docstring) │
│    args_schema   → Input schema (Pydantic model)                 │
│    invoke()      → Execute the tool                              │
│    ainvoke()     → Async execution                               │
│                                                                  │
│  TYPES OF TOOLS:                                                 │
│    Simple Function  → @tool decorator                            │
│    Class-based      → Inherit from BaseTool                      │
│    Built-in         → LangChain's pre-built tools                │
│    Custom           → Your own business logic                    │
│                                                                  │
│  COMMON TOOLS:                                                   │
│    - WikipediaQueryRun      → Wikipedia search                   │
│    - TavilySearchResults    → Web search                         │
│    - PythonREPL             → Code execution                     │
│    - SQLDatabase            → Database queries                   │
│    - Gmail integration      → Send emails                        │
│    - Custom: Database, API, File, Email, etc.                    │
│                                                                  │
│  AGENTS + TOOLS:                                                 │
│    Agent = LLM that decides WHICH tools to use                   │
│                                                                  │
│    Agent Loop:                                                   │
│      Think → Use Tool → Observe → Think → Answer                 │
│      (ReAct: Reasoning + Acting)                                 │
│                                                                  │
│  BEST PRACTICES:                                                 │
│    ✅ Clear descriptions → LLM knows when to use                 │
│    ✅ Well-defined parameters → Easy to parse                    │
│    ✅ Error handling → Graceful failures                         │
│    ✅ Limited tools → Avoid decision paralysis                   │
│    ✅ Security checks → Validate inputs                          │
│    ✅ Rate limiting → Prevent abuse                              │
│                                                                   │
│  COMMON PITFALLS:                                                 │
│    ❌ Too many tools → LLM gets confused                          │
│    ❌ Vague descriptions → LLM doesn't use them                   │
│    ❌ No error handling → System crashes on bad input             │
│    ❌ Slow tools → Agent times out                                │
│    ❌ Security gaps → Attackers exploit tools                     │
│                                                                    │
│  GOLDEN RULE:                                                      │
│  "Give LLMs clear, focused tools with sharp descriptions.          │
│   The better you describe what a tool does, the better the         │
│   agent uses it. Fewer, well-designed tools > many confusing tools."│
└─────────────────────────────────────────────────────────────────────┘

"""